# Actividad 1 · Del análisis exploratorio a la predicción

**Tiempo estimado:** 20 minutos  
**Recorrido:** SQL + Python

En la Unidad 4 usamos el análisis exploratorio para describir patrones del dataset de churn. En esta actividad vamos a dar el paso siguiente: transformar esos hallazgos en una pregunta predictiva.

> El notebook prepara automáticamente el entorno de trabajo, descarga el dataset y crea la conexión necesaria para realizar las consultas. En la primera ejecución puede instalar algún paquete que no esté disponible en el entorno.


## Describir y predecir no son lo mismo

| Enfoque | Pregunta principal | Resultado esperado |
|---|---|---|
| Análisis exploratorio | ¿Qué ocurrió y qué patrones aparecen? | Tablas, gráficos e interpretaciones |
| Modelado predictivo | ¿Qué podría ocurrir en un caso nuevo? | Una predicción o una probabilidad por caso |

El modelado no reemplaza al análisis exploratorio: se apoya en él. Antes de predecir necesitamos comprender el problema, la procedencia de los datos y la variable que queremos anticipar.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

print("Preparando el entorno...")

for package in ["duckdb", "pandas"]:
    if importlib.util.find_spec(package) is None:
        print(f"Instalando {package}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import duckdb
import pandas as pd

print("Paquetes preparados correctamente.")

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH = Path("Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    print("Descargando dataset...")
    urlretrieve(DATA_URL, DATA_PATH)

churn = pd.read_csv(DATA_PATH)

required_columns = {"Churn", "tenure", "MonthlyCharges", "TotalCharges"}
if not required_columns.issubset(churn.columns):
    raise ValueError("El dataset descargado no contiene las columnas esperadas.")

churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")

# Si la celda se ejecuta otra vez, cerramos primero una conexión previa.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(":memory:")
con.register("churn_df", churn)
con.execute("CREATE OR REPLACE TABLE churn AS SELECT * FROM churn_df")

clientes = f"{len(churn):,}".replace(",", ".")
print(f"Entorno listo: {clientes} clientes y {churn.shape[1]} variables.")


## Recuperamos una síntesis del caso con SQL


In [ ]:
resumen = con.sql("""
SELECT
    Churn AS abandono,
    COUNT(*) AS clientes,
    ROUND(AVG(tenure), 1) AS antiguedad_promedio,
    ROUND(AVG(MonthlyCharges), 2) AS cargo_mensual_promedio
FROM churn
GROUP BY Churn
ORDER BY Churn
""").df()
resumen


## Del hallazgo a la pregunta predictiva

Un hallazgo descriptivo que surge de la síntesis anterior podría ser: **“los clientes que abandonaron el servicio presentan, en promedio, menor antigüedad que quienes no abandonaron”**. Una pregunta predictiva sería: **¿qué clientes actuales tienen mayor probabilidad de abandonar el servicio?**

Respondé brevemente:

1. ¿Cuál sería la variable objetivo del problema?
2. ¿El resultado que queremos predecir es una categoría o un valor numérico?
3. ¿Qué variables del dataset podrían funcionar como predictoras?
4. Si existiera una variable que solo se conociera después del abandono —por ejemplo, una fecha de baja registrada posteriormente—, ¿debería utilizarse para predecir? ¿Por qué?


## Cierre

Al finalizar deberías poder distinguir una conclusión descriptiva de una pregunta predictiva e identificar, de manera inicial, la variable objetivo y posibles predictores.


In [ ]:
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass
